# A sample to label by hand

A hundred replies from each model, drawn to be labelled by a person, so that
the classifier can be measured against something other than itself.

Every number in the results passes through the classifier, so its accuracy is
the accuracy of the study. The only way to establish that is to label a sample
by hand and compare. This notebook draws the sample and, once the labelling is
done, reads it back and reports the agreement.

**The same hundred prompts for every model.** A prompt is only eligible if
every model answered it, so the files line up row for row. That is more work to
select and worth it twice over: the models become comparable on identical items,
and a disagreement between the classifier and you can be read across all six at
once rather than one file at a time.

Replicate 1 only, and nothing blocked, errored or empty. A reply that never
arrived cannot be labelled, and including it would put the classifier's handling
of an absent reply into a figure meant to measure its reading of a present one.

A CSV per model with the label columns blank, and one more holding the replies
a provider withheld. Those are already labelled and are not for reading: nobody
decided them, so a label from you and a label from the classifier would agree by
construction and measure nothing.

In [1]:
# Import the libraries
import sys
from pathlib import Path

import pandas as pd

In [2]:
# Set the working directory to the project root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

/Users/rinlobachevskii/Desktop/Git/Thesis


In [3]:
# Import the pipeline
%load_ext autoreload
%autoreload 2

import settings
import utils

ANNOTATION_DIR = settings.RESULTS_DIR / 'annotation'
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_colwidth', 70)
print('Ready')

Ready


## Read what was collected

Replicate 1 of every model, with the replies that never arrived left out.

In [4]:
collected = []
for path in sorted(settings.ADAPTATION_DIR.glob('*.jsonl')):
    frame = utils.read_lines(path)
    if not frame.empty:
        collected.append(frame)
if not collected:
    raise SystemExit(f'Nothing collected in {settings.ADAPTATION_DIR}')

replies = pd.concat(collected, ignore_index=True)
replies['response'] = replies['response'].astype(str)
replies['blocked'] = replies['blocked'].astype(str)

# What the provider withheld, at whatever replicate it happened. A prompt whose
# third draw was blocked still has a first and a second, so only the blocked
# draw is treated this way and the others stay eligible like any other.
withheld = replies[replies['blocked'].str.strip() != ''].copy()

first = replies[replies['replicate'].astype(str) == '1']
answered = first[(first['response'].str.strip() != '')
                 & (first['blocked'].str.strip() == '')
                 & (first['error'].astype(str).str.strip() == '')]

print(f'{len(replies):,} replies, {len(withheld)} withheld by a provider')
display(withheld.groupby(['model', 'blocked']).size().rename('replies').to_frame())
print(f'\n{len(answered):,} answered at replicate 1, of {len(first):,}')

46,800 replies, 160 withheld by a provider


replies
model                     blocked                    
claude-haiku-4-5-20251001 CONTENT_FILTER            1
gemini-3.5-flash-lite     PROHIBITED_CONTENT      158
                          RECITATION                1


15,547 answered at replicate 1, of 15,600


## Only prompts every model answered

The sample is the same for all of them, so a prompt one model failed to answer
is dropped for all. Gemini's blocked prompts are most of what goes here, and
losing them from the labelled sample does not lose them from the results: they
are reported as their own outcome elsewhere.

In [5]:
answered_by = answered.groupby('prompt_id')['model'].nunique()
models = answered.groupby('model').ngroups
shared = set(answered_by[answered_by == models].index)

print(f'{len(shared):,} prompts answered at replicate 1 by all {models} models, '
      f'of {answered["prompt_id"].nunique():,}')
print('The rest had at least one model that did not answer, and are left out of')
print('the shared draw so that the files line up. They are not lost: a prompt')
print('withheld at one replicate is in the section above at that replicate.')

2,547 prompts answered at replicate 1 by all 6 models, of 2,600
The rest had at least one model that did not answer, and are left out of
the shared draw so that the files line up. They are not lost: a prompt
withheld at one replicate is in the section above at that replicate.


## Draw five hundred

Spread across scenario type, domain and condition, so that the labelled sample
looks like the benchmark rather than like whichever rows sorted first. The draw
is seeded, so re-running this gives the same five hundred and a labelling
session can be resumed.

In [6]:
HOW_MANY = 100

prompts = utils.read_table(settings.PROMPTS_PATH)
benchmark = utils.read_table(settings.BENCHMARK_PATH)
frame = (prompts[prompts['prompt_id'].isin(shared)]
         .merge(benchmark[['scenario_id', 'domain', 'scenario_type', 'category']],
                on='scenario_id'))

# proportional to the benchmark, so the sample is a small copy of it rather than
# an equal slice of unequal strata
frame['cell'] = (frame['scenario_type'] + ' | ' + frame['domain']
                 + ' | ' + frame['band'].replace('', 'none'))
share = HOW_MANY / len(frame)
chosen = (frame.sample(frac=1, random_state=settings.SEED)
          .groupby('cell', group_keys=False)
          .apply(lambda cell: cell.head(max(1, round(len(cell) * share))),
                 include_groups=False)
          .head(HOW_MANY))

print(f'{len(chosen)} prompts drawn from {frame["cell"].nunique()} cells\n')
display(pd.crosstab(chosen['scenario_type'], chosen['band'].replace('', 'none')))
print()
display(chosen.groupby('domain').size().rename('prompts').to_frame().T)

100 prompts drawn from 175 cells



band,Adolescent,Adult,Child,Minor,none
scenario_type,,,,,
Age Restricted,5,5,5,5,5
Benign,10,10,10,10,10
Harmful,5,5,5,5,5


domain,Abuse & Hate,Body Image,Bullying,Dangerous Challenges,Eating Disorders,Emotional Dependency,Harmful Substances,Self-Harm & Suicide,Sexual Content,Violence
prompts,10,15,10,15,10,5,10,5,10,10


## Write one file per model

Same rows, same order, one model each. The label columns are empty and are for
you: fill in `answer` and the five properties as you read, leaving anything you
are unsure of blank rather than guessing, since a doubtful label is worse than a
missing one when the point is to measure a classifier against it.

`expected_answer` is included because it is part of the benchmark, not a hint.
Label what the reply did, not what it should have done.

In [7]:
LABEL_COLUMNS = ['answer'] + [settings.measure_column(name)
                              for name in settings.SAFETY]
# The same order as everything else the pipeline writes: model, prompt and
# replicate identify a row, then who judged it, then what was said and what was
# decided. A labelled sheet is a judgement file with the text kept beside it, so
# the two line up column for column across the part they share.
#
# Nothing derived or derivable is stored. The scenario type, domain, category
# and condition are in the prompt identifier and in benchmark.csv; the expected
# answer is in prompts.csv. Keeping them out is not only tidiness: an expected
# answer beside the label would nudge it, and a stored one would go stale the
# moment a stratum is revised. What a reply did is an observation; whether it
# should have is a comparison, and the comparison is made where the two meet.
BEFORE = ['model', 'prompt_id', 'replicate', 'judge', 'prompt', 'response']
SHEET_COLUMNS = BEFORE

facts = prompts[['prompt_id', 'prompt']]
WHO = 'human'          # whose labels these are, beside the classifier's name
drawn = set(chosen['prompt_id'])


# Define function to lay out one model's rows in the order they will be read.
# The reason a provider gave is carried through, because a prompt can be
# withheld at one replicate and answered at another, and the row is about the
# replicate rather than the prompt.
def sheet_for(rows):
    sheet = rows.merge(facts, on='prompt_id')
    sheet['note'] = sheet['blocked'].astype(str)
    sheet['judge'] = WHO
    for column in LABEL_COLUMNS:
        sheet[column] = ''
    return (sheet[BEFORE + LABEL_COLUMNS + ['note']]
            .sort_values(['prompt_id', 'replicate']))


# The withheld replies are recorded once, together, and are not for labelling.
# Nobody decided them: the provider did, so a label from you and a label from
# the classifier would agree by construction and tell you nothing. They are here
# so the set is visible and countable, not so it can be read.
withheld_sheet = sheet_for(withheld)
if len(withheld_sheet):
    withheld_sheet['answer'] = settings.BLOCKED
    path = ANNOTATION_DIR / 'blocked.csv'
    withheld_sheet.to_csv(path, index=False)
    print(f'  {len(withheld_sheet):>4} withheld, recorded in {path.name}, '
          f'not for labelling')
    print(f'       {withheld_sheet.groupby("model").size().to_dict()}\n')

for model, group in replies.groupby('model'):

    # then the shared draw, at replicate 1, for you to read and label
    to_label = sheet_for(group[(group['prompt_id'].isin(drawn))
                               & (group['replicate'].astype(str) == '1')
                               & (group['blocked'].str.strip() == '')])
    to_label['note'] = ''

    path = ANNOTATION_DIR / f'{utils.model_slug(model)}.csv'
    to_label.to_csv(path, index=False)
    print(f'  {model:<28} {len(to_label):>4} to label  {path.name}')

print(f'\nWritten to {ANNOTATION_DIR}')
print(f'Columns to fill: {", ".join(LABEL_COLUMNS)}')
print(f'The {settings.BLOCKED} rows are filled in already. Leave them.')

   160 withheld, recorded in blocked.csv, not for labelling
       {'claude-haiku-4-5-20251001': 1, 'gemini-3.5-flash-lite': 159}

  claude-haiku-4-5-20251001     100 to label  claude-haiku-4-5-20251001.csv
  deepseek-v4-flash             100 to label  deepseek-v4-flash.csv
  gemini-3.5-flash-lite         100 to label  gemini-3.5-flash-lite.csv
  gemma4:31b-cloud              100 to label  gemma4-31b-cloud.csv
  gpt-5.6-luna                  100 to label  gpt-5.6-luna.csv
  mistral-small-2603            100 to label  mistral-small-2603.csv

Written to /Users/rinlobachevskii/Desktop/Git/Thesis/results/annotation
Columns to fill: answer, guidance, boundary, signposting, overreliance, privacy_violation
The Blocked rows are filled in already. Leave them.


## How much labelling this is

Six files of a hundred is six hundred judgements, which is a sensible size for
an agreement figure: the confidence interval on a kappa barely narrows above two
or three hundred, and six hundred spread across six models gives each of them a
hundred of its own.

The cell below reads back whatever has been filled in, so the agreement can be
computed on a partial sheet and again when more is done.

In [8]:
labelled, empty = [], []
for path in sorted(ANNOTATION_DIR.glob('*.csv')):
    if path.stem == 'blocked':
        continue
    sheet = pd.read_csv(path, dtype=str, keep_default_na=False)
    done = sheet[sheet['answer'].str.strip() != '']
    (labelled if len(done) else empty).append((path.stem, sheet, done))

for stem, sheet, done in labelled:
    print(f'  {stem:<28} {len(done):>4} of {len(sheet)} labelled '
          f'({len(done) / len(sheet):>4.0%})')
for stem, sheet, _ in empty:
    print(f'  {stem:<28}    0 of {len(sheet)} labelled')
if not labelled:
    print('\nNothing labelled yet. Fill in the answer column and run this again.')

  claude-haiku-4-5-20251001       0 of 100 labelled
  deepseek-v4-flash               0 of 100 labelled
  gemini-3.5-flash-lite           0 of 100 labelled
  gemma4-31b-cloud                0 of 100 labelled
  gpt-5.6-luna                    0 of 100 labelled
  mistral-small-2603              0 of 100 labelled

Nothing labelled yet. Fill in the answer column and run this again.


## Agreement with the classifier

Once the judgements exist, this compares them row by row. Two figures matter and
they say different things.

**Raw agreement** is the share the classifier and the person called the same. It
is easy to read and it flatters any measure where one value dominates: a
property that is No ninety five per cent of the time agrees ninety five per cent
of the time by chance alone.

**Cohen's kappa** removes that, and it is the figure to report. Above about 0.8
is strong, 0.6 to 0.8 is usable, below 0.4 means the classifier and the person
are not applying the same definition, and the definition is what needs fixing.

In [9]:
# Define function to compare two columns of labels, correcting for the
# agreement that would happen by chance given how often each value occurs
def kappa(left, right):
    pairs = [(a, b) for a, b in zip(left, right)
             if str(a).strip() and str(b).strip()]
    if not pairs:
        return None, 0
    agreed = sum(a == b for a, b in pairs) / len(pairs)
    values = {value for pair in pairs for value in pair}
    expected = sum(
        (sum(a == value for a, _ in pairs) / len(pairs))
        * (sum(b == value for _, b in pairs) / len(pairs))
        for value in values)
    if expected >= 1:
        return 1.0, len(pairs)
    return round((agreed - expected) / (1 - expected), 3), len(pairs)


judgements = []
for path in sorted(settings.JUDGEMENTS_DIR.glob('*.jsonl')):
    frame = utils.read_lines(path)
    if not frame.empty:
        judgements.append(frame)

if not judgements or not labelled:
    print('Needs both the judgements and some labels. Run the judge with')
    print('    python scripts/evaluate.py --backend ollama --workers 8')
else:
    judged = pd.concat(judgements, ignore_index=True)
    judged = judged[judged['replicate'].astype(str) == '1']
    rows = []
    for stem, sheet, done in labelled:
        pair = done.merge(judged, on=['prompt_id', 'model'],
                          suffixes=('_person', '_judge'))
        for column in LABEL_COLUMNS:
            score, n = kappa(pair[f'{column}_person'], pair[f'{column}_judge'])
            same = (pair[f'{column}_person'] == pair[f'{column}_judge']).mean()
            rows.append({'model': stem, 'measure': column, 'n': n,
                         'agreed': round(same, 3), 'kappa': score})
    agreement = pd.DataFrame(rows)
    display(agreement.pivot(index='measure', columns='model', values='kappa'))
    print('\nraw agreement, for comparison:')
    display(agreement.pivot(index='measure', columns='model', values='agreed'))

Needs both the judgements and some labels. Run the judge with
    python scripts/evaluate.py --backend ollama --workers 8


## Where they differ

The rows to read. A classifier that disagrees at random is noisy; one that
disagrees the same way every time has learned a different rule from the one you
meant, and the policy is what to change.

In [10]:
if labelled and judgements:
    for stem, sheet, done in labelled:
        pair = done.merge(judged, on=['prompt_id', 'model'],
                          suffixes=('_person', '_judge'))
        differ = pair[pair['answer_person'] != pair['answer_judge']]
        if differ.empty:
            continue
        print('=' * 74)
        print(f'{stem}: {len(differ)} of {len(pair)} answers differ')
        print('=' * 74)
        for row in differ.head(10).itertuples():
            print(f'\n  {row.prompt_id}  {row.scenario_type}, {row.condition}')
            print(f'    asked   {row.request}')
            print(f'    replied {str(row.response)[:170]}...')
            print(f'    you said {row.answer_person}, '
                  f'the classifier said {row.answer_judge}')
            if str(getattr(row, "note", "")).strip():
                print(f'    your note: {row.note}')

## Then

Report the kappa per measure in the methods, alongside the size of the labelled
sample and who labelled it. If a measure comes back low, the definition in
`build_policy` is what to change, and everything judged under the old wording
has to be judged again, which is the reason to do this before the full pass
rather than after.